# Session 9 — A Question-Answering Pipeline

**MA553 · Session 9 workbook**

Name:

---

Today we build a small system in four steps, and we measure each step. The slides and
this notebook alternate: after each block of slides, work through the matching part.

| Part | After the slides on | What you measure |
|---|---|---|
| 1 | the pipeline and retrieval | how often search finds the right paragraph |
| 2 | the reading step | accuracy of each step, and where the errors come from |
| 3 | cost and latency | time and length of the prompt, against accuracy |
| 4 | monitoring | what changes when questions move to new topics |

The system uses the two models you already have:
- the search model from Session 6 (`all-MiniLM-L6-v2`), in `../lab5/model/`;
- the language model from Session 7 (`Qwen2.5-1.5B-Instruct`), in `../lab6/model/`.

If your models are somewhere else, change the two paths in the setup cell.

Cells marked **Write this yourself** are short. The **Answer here** cells are discussed in
class.

# Part 1 — Retrieval

## Setup

Run this. Loading both models takes about 30 seconds.

In [ ]:
import json, re, string, time
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

SEARCH_MODEL = "../lab5/model/all-MiniLM-L6-v2"
LANGUAGE_MODEL = "../lab6/model/qwen2.5-1.5b-instruct"

search = SentenceTransformer(SEARCH_MODEL, device="cpu")
tok = AutoTokenizer.from_pretrained(LANGUAGE_MODEL)
lm = AutoModelForCausalLM.from_pretrained(LANGUAGE_MODEL, dtype=torch.bfloat16)
lm.eval()
print("Models loaded.")

## The data

SQuAD is a set of questions about Wikipedia paragraphs. Each question has a known answer,
taken from its paragraph.

This cell keeps 40 Wikipedia articles as the **collection** that the system can search, and
holds 8 articles back for Part 4. It then draws 40 questions about the collection.

Run this.

In [ ]:
rng = np.random.default_rng(9)
articles = json.load(open("squad-dev-v1.1.json"))["data"]
order = rng.permutation(len(articles))
held_out = set(order[:8].tolist())

paragraphs = []          # the collection
questions = []           # questions about the collection
new_questions = []       # questions about the 8 held-out articles (Part 4)
for a, article in enumerate(articles):
    for p in article["paragraphs"]:
        if a not in held_out:
            paragraphs.append(p["context"])
        for q in p["qas"]:
            item = {"question": q["question"],
                    "answers": [x["text"] for x in q["answers"]],
                    "paragraph": p["context"],
                    "pid": len(paragraphs) - 1 if a not in held_out else None}
            (new_questions if a in held_out else questions).append(item)

questions = [questions[i] for i in rng.choice(len(questions), 40, replace=False)]
new_questions = [new_questions[i] for i in rng.choice(len(new_questions), 20, replace=False)]
print("Paragraphs in the collection:", len(paragraphs))
print()
print("Question:", questions[0]["question"])
print("Answer:  ", questions[0]["answers"][0])

## Search

This cell turns every paragraph into a vector, as in Session 6, and finds the 5 closest
paragraphs for each question. For each question, `top` holds the paragraph numbers, best
first, and `scores` holds their cosine similarities.

Run this. Embedding the collection takes about 30 seconds.

In [ ]:
t = time.perf_counter()
P = search.encode(paragraphs, normalize_embeddings=True, batch_size=64)
print(f"Collection embedded in {time.perf_counter() - t:.0f} s")

def retrieve(items, k=5):
    Q = search.encode([x["question"] for x in items], normalize_embeddings=True)
    S = Q @ P.T
    for x, row in zip(items, S):
        best = np.argsort(-row)[:k]
        x["top"] = best.tolist()
        x["scores"] = row[best].tolist()

retrieve(questions)
print("Best paragraph for the first question:", questions[0]["top"][0],
      "  correct paragraph:", questions[0]["pid"])

## 1 — Recall ✏️ Write this yourself

**Recall@k** is the share of questions whose correct paragraph (`pid`) is among the top
`k` results.

**Goal.** Print recall@1, recall@3 and recall@5 for the 40 questions.

**Expected.** recall@1 62%, recall@3 85%, recall@5 88%.

In [ ]:
# Your code here.

# Part 2 — Reading

## The reading step

The function `answer(question, passages)` gives the language model some passages and the
question, and returns its short answer. It also returns the time it took and the number of
tokens in the prompt.

An answer counts as **correct** if it contains one of the known answers, ignoring case,
punctuation, and the words "a", "an", "the".

Run this.

In [ ]:
INSTRUCTION = ("Answer the question using only the passages below. Answer with a short phrase. "
               "If the passages do not contain the answer, answer exactly: not in the passages.")

def answer(question, passages):
    text = INSTRUCTION + "\n\n" + "\n\n".join(
        f"Passage {i + 1}: {p}" for i, p in enumerate(passages)) + f"\n\nQuestion: {question}"
    x = tok.apply_chat_template([{"role": "user", "content": text}], add_generation_prompt=True,
                                return_tensors="pt", return_dict=True)
    t = time.perf_counter()
    with torch.no_grad():
        out = lm.generate(**x, do_sample=False, max_new_tokens=24)
    seconds = time.perf_counter() - t
    reply = tok.decode(out[0, x["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return reply, seconds, x["input_ids"].shape[1]

def normalize(s):
    s = "".join(c for c in s.lower() if c not in string.punctuation)
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", s).split())

def is_correct(reply, answers):
    return any(normalize(a) and normalize(a) in normalize(reply) for a in answers)

def run(items, passages_for):
    rows = []
    for x in items:
        reply, seconds, tokens = answer(x["question"], passages_for(x))
        rows.append({"reply": reply, "correct": is_correct(reply, x["answers"]),
                     "abstain": "not in the passages" in reply.lower(),
                     "seconds": seconds, "tokens": tokens})
    return rows

x = questions[0]
print(x["question"], "->", answer(x["question"], [x["paragraph"]])[0])

## Two ways to read

- **Given paragraph:** the model reads the correct paragraph. This measures the reading
  step alone.
- **Retrieved paragraph:** the model reads the top-1 result of search. This measures the
  whole pipeline.

Run this. It takes about 2 to 4 minutes.

In [ ]:
given = run(questions, lambda x: [x["paragraph"]])
top1 = run(questions, lambda x: [paragraphs[x["top"][0]]])
print("Done.")

## 2 — Where the errors come from ✏️ Write this yourself

**Goal.**
1. Print the accuracy with the given paragraph and with the retrieved paragraph.
2. For the retrieved paragraph, count the questions in each of four groups: search found
   the right paragraph or not (`hit`), and the answer is correct or not.

**Expected.** Accuracy 80% with the given paragraph and 60% with the retrieved
paragraph. Groups: found and correct 22, found and wrong 3, missed and correct 2, missed and
wrong 13.

In [ ]:
# Your code here.

**Answer here.** (1) Multiply recall@1 by the accuracy with the given paragraph. Is the
result close to the accuracy of the whole pipeline? (2) Which step causes more of the
failures?

*Answer here.*

# Part 3 — Cost and latency

## Three paragraphs instead of one

Giving the model the top 3 results raises the chance that the right paragraph is there,
but the prompt is about three times longer.

Run this. It takes about 3 to 5 minutes.

In [ ]:
top3 = run(questions, lambda x: [paragraphs[j] for j in x["top"][:3]])
print("Done.")

## 3 — Accuracy against cost ✏️ Write this yourself

**Goal.** For `top1` and `top3`, print the accuracy, the average number of tokens in the
prompt, and the average seconds per question.

**Expected.** top-1: accuracy 60%, about 217 tokens. top-3: accuracy 72%, about 530
tokens. The seconds depend on your laptop; top-3 takes about 1.5 times as long as top-1.

In [ ]:
# Your code here.

**Answer here.** A help desk receives 10,000 questions a day. Which version would you
deploy, and what would you need to know to decide?

*Answer here.*

# Part 4 — Monitoring

## New topics

After deployment, users start asking about topics that are not in the collection. Here,
these are the 20 questions about the 8 held-out articles.

In practice, we do not know the correct answers to new questions. Therefore, we look for
measures that need no correct answers: the best retrieval score, and how often the model answers
"not in the passages".

Run this. It takes about 2 minutes.

In [ ]:
retrieve(new_questions)
new_top3 = run(new_questions, lambda x: [paragraphs[j] for j in x["top"][:3]])
print("Done.")

## 4 — Measures that need no correct answers ✏️ Write this yourself

**Goal.** For the 40 questions about the collection (`questions`, `top3`) and the 20 new
questions (`new_questions`, `new_top3`), print:
1. the average best retrieval score (`x["scores"][0]`);
2. the share of answers that are "not in the passages";
3. the accuracy.

**Expected.** known topics: best score 0.61, "not in the passages" 8%, accuracy 72%.
new topics: best score 0.38, "not in the passages" 80%, accuracy 0%.

In [ ]:
# Your code here.

**Answer here.** Which of these three numbers could you track every day in a deployed
system, and why?

*Answer here.*

---
**Before you leave.** Save the notebook and submit it through Moodle.